In [1]:
import pandas as pd

data = pd.read_csv("iris_demo.csv")
print(data.head())
print(data.shape)
print(data.isna().sum())
print(data["species"].value_counts())

   sepal_length_cm  sepal_width_cm  petal_length_cm  petal_width_cm species
0              5.1             3.5              1.4             0.2  setosa
1              4.9             3.0              1.4             0.2  setosa
2              4.7             3.2              1.3             0.2  setosa
3              4.6             3.1              1.5             0.2  setosa
4              5.0             3.6              1.4             0.2  setosa
(150, 5)
sepal_length_cm    0
sepal_width_cm     0
petal_length_cm    0
petal_width_cm     0
species            0
dtype: int64
species
setosa        50
versicolor    50
virginica     50
Name: count, dtype: int64


I loaded the iris dataset to check the flower measurements and look for missing values. The dataset has 150 flowers, three species, and no missing values.

In [2]:
from sklearn.model_selection import train_test_split

X = data.drop(columns="species")
y = data["species"]

X_develop, X_test, y_develop, y_test = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=42
)

X_train, X_valid, y_train, y_valid = train_test_split(
    X_develop, y_develop, test_size=0.25, stratify=y_develop, random_state=42
)

print(len(X_train), len(X_valid), len(X_test))

90 30 30


I split the data into training, validation, and testing groups. This helps me train the model and check how well it works on new data.

In [3]:
from sklearn.dummy import DummyClassifier
from sklearn.metrics import accuracy_score

baseline = DummyClassifier(strategy="most_frequent")
baseline.fit(X_train, y_train)

baseline_pred = baseline.predict(X_valid)

print("Baseline accuracy:", accuracy_score(y_valid, baseline_pred))

Baseline accuracy: 0.3333333333333333


I created a simple model that guesses the most common flower. This gives me something to compare my neural network against.

In [4]:
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPClassifier

model = make_pipeline(
    StandardScaler(),
    MLPClassifier(hidden_layer_sizes=(8,), solver="lbfgs", max_iter=1000, random_state=42)
)

model.fit(X_train, y_train)

valid_pred = model.predict(X_valid)

print("Validation accuracy:", accuracy_score(y_valid, valid_pred))

Validation accuracy: 0.9333333333333333


I trained a neural network with eight neurons to predict flower species. I also scaled the data to help the model learn better.

In [5]:
comparison = pd.DataFrame({
    "actual": y_valid.to_numpy(),
    "predicted": valid_pred
})

print(comparison.head())
print("Validation errors:", (comparison.actual != comparison.predicted).sum())

       actual   predicted
0   virginica   virginica
1  versicolor  versicolor
2  versicolor  versicolor
3      setosa      setosa
4      setosa      setosa
Validation errors: 2


I compared the model's predictions to the actual flower species. This helps me see how many mistakes the model made.

In [6]:
smaller = make_pipeline(
    StandardScaler(),
    MLPClassifier(hidden_layer_sizes=(4,), solver="lbfgs", max_iter=1000, random_state=42)
)

smaller.fit(X_train, y_train)

small_pred = smaller.predict(X_valid)

print("Four-neuron validation accuracy:", accuracy_score(y_valid, small_pred))
print("Eight-neuron validation accuracy:", accuracy_score(y_valid, valid_pred))

Four-neuron validation accuracy: 0.9666666666666667
Eight-neuron validation accuracy: 0.9333333333333333


I tested a smaller network with four neurons and compared it to the one with eight. This helps me see which model performs better.

In [7]:
chosen_size = 4 if accuracy_score(y_valid, small_pred) >= accuracy_score(y_valid, valid_pred) else 8

final_model = make_pipeline(
    StandardScaler(),
    MLPClassifier(hidden_layer_sizes=(chosen_size,), solver="lbfgs", max_iter=1000, random_state=42)
)

final_model.fit(X_develop, y_develop)

test_pred = final_model.predict(X_test)

print("Final test accuracy:", accuracy_score(y_test, test_pred))

new_flower = pd.DataFrame([[5.1, 3.5, 1.4, 0.2]], columns=X.columns)

print("New flower prediction:", final_model.predict(new_flower)[0])

Final test accuracy: 0.9
New flower prediction: setosa


I used the better model with four neurons and tested it on new data. I also used it to predict the species of a new flower.

A weight controls how much an input affects the result. A bias helps adjust the output, and an activation function decides how the neuron responds. A hidden layer helps the network recognize patterns and make better predictions. The baseline gives me something simple to compare my model against. The test set helps me see how well my model works on data it hasn't seen before.